In [ ]:
# OpenCV related import
import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image,display

## Session 7

1.create folders to store the images

In [ ]:
import os

# Create Main dataset folder
DATASET_FOLDER = "traffic_sign_dataset"

In [ ]:
# Create LEFT_FOLDER path
DATASET_FOLDER = "traffic_sign_dataset"
LEFT_FOLDER=os.path.join(DATASET_FOLDER,"LEFT")

# Create the real folder
os.makedirs(LEFT_FOLDER,exist_ok=True)

In [ ]:
import os

# Main dataset folder
DATASET_FOLDER = "traffic_sign_dataset"

# Four class folders
LEFT_FOLDER = os.path.join(DATASET_FOLDER, "LEFT")
RIGHT_FOLDER = os.path.join(DATASET_FOLDER, "RIGHT")
STOP_FOLDER = os.path.join(DATASET_FOLDER, "STOP")
NO_SIGN_FOLDER = os.path.join(DATASET_FOLDER, "NO_SIGN")

# Create the folders
os.makedirs(LEFT_FOLDER, exist_ok=True)
os.makedirs(RIGHT_FOLDER, exist_ok=True)
os.makedirs(STOP_FOLDER, exist_ok=True)
os.makedirs(NO_SIGN_FOLDER, exist_ok=True)

print("Dataset folders are ready.")

2. collect the images

In [ ]:
# frame.shape
import cv2
import time
from IPython.display import display, Image

camera = cv2.VideoCapture(0, cv2.CAP_AVFOUNDATION)
time.sleep(1)

if camera.isOpened():
    frame_ok, frame = camera.read()

    if frame_ok:
        jpeg_ok, jpeg = cv2.imencode(".jpg", frame)

        if jpeg_ok:
            display(Image(data=jpeg.tobytes()))
            print("Full shape:", frame.shape)

            frame_height, frame_width = frame.shape[:2]
            print("Height:", frame_height)
            print("Width:", frame_width)
else:
    print("The camera could not be opened.")

camera.release()

In [ ]:
import cv2
import time
import os
from IPython.display import display, Image


# Choose which category to collect
save_folder = LEFT_FOLDER
class_name = "LEFT"

total_images = 30
save_interval = 1


# Open the MacBook camera
camera = cv2.VideoCapture(0, cv2.CAP_AVFOUNDATION)

# Windows:
# camera = cv2.VideoCapture(0, cv2.CAP_DSHOW)


if not camera.isOpened():
    print("The camera could not be opened.")

else:
    print(f"Collecting {class_name} images.")
    print("Place the sign inside the green box.")

    time.sleep(5)

    collected = 0
    last_save_time = 0
    handle = None

    while collected < total_images:

        frame_ok, frame = camera.read()

        if not frame_ok:
            print("The camera could not capture an image.")
            break

        # Get the frame size
        frame_height, frame_width = frame.shape[:2]

        # Create a 300 × 300 box in the centre
        box_size = 300

        x1 = (frame_width - box_size) // 2
        y1 = (frame_height - box_size) // 2
        x2 = x1 + box_size
        y2 = y1 + box_size

        # Crop the image inside the box
        scanning_area = frame[y1:y2, x1:x2] # collect images before drawing the green box command below to prevent to collecting the green box.

        current_time = time.time()

        # Save one image every 1 second
        if current_time - last_save_time >= save_interval:

            filename = os.path.join(
                save_folder,
                f"{class_name.lower()}_{collected}.jpg"
            )

            saved_ok = cv2.imwrite(filename, scanning_area)

            if saved_ok:
                collected += 1
                last_save_time = current_time
                print(f"Saved image {collected}")

        # Draw the green scanning box
        cv2.rectangle(
            frame,
            (x1, y1),
            (x2, y2),
            (0, 255, 0),
            3
        )

        # Show the number of collected images
        cv2.putText(
            frame,
            f"{class_name}: {collected}/{total_images}",
            (x1, y1 - 15),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 0),
            2
        )

        # Show the live video in Jupyter Notebook
        jpeg_ok, jpeg = cv2.imencode(".jpg", frame)

        if jpeg_ok:

            if handle is None:
                handle = display(
                    Image(data=jpeg.tobytes()),
                    display_id=True
                )

            else:
                handle.update(
                    Image(data=jpeg.tobytes())
                )

        time.sleep(0.03)

    camera.release()

    print(f"Finished collecting {collected} {class_name} images.")
    print("Camera released.")

In [ ]:
import cv2
import time
import os
import glob
from IPython.display import display, Image


def collect_images(class_name, save_folder, total_images=30):

    save_interval = 1

    # Open the MacBook camera
    camera = cv2.VideoCapture(0, cv2.CAP_AVFOUNDATION)

    # Windows:
    # camera = cv2.VideoCapture(0, cv2.CAP_DSHOW)

    if not camera.isOpened():
        print("The camera could not be opened.")
        camera.release()
        return

    # Start counting from the images that are already saved, so running this
    # again ADDS new images instead of writing over the old ones.
    # This is what makes it possible to collect 10 images from one sign
    # picture, change the picture, and collect 10 more.
    collected = len(glob.glob(os.path.join(save_folder, "*.jpg")))
    target = collected + total_images

    print(f"Collecting {class_name} images.")
    print(f"The folder already has {collected} images.")
    print("Place the sign inside the green box.")

    time.sleep(3)

    last_save_time = 0
    failed_saves = 0
    handle = None

    while collected < target:

        frame_ok, frame = camera.read()

        if not frame_ok:
            print("The camera could not capture an image.")
            break

        # Get the frame size
        frame_height, frame_width = frame.shape[:2]

        # Create a 300 x 300 box in the centre.
        # The camera records at 1920 x 1080, so this box covers about one
        # sixth of the width. Every training image is cropped at this size.
        box_size = 300

        x1 = (frame_width - box_size) // 2
        y1 = (frame_height - box_size) // 2
        x2 = x1 + box_size
        y2 = y1 + box_size

        # Crop the image inside the box BEFORE the green box is drawn,
        # so the green box is never saved into the training picture
        scanning_area = frame[y1:y2, x1:x2]

        current_time = time.time()

        # Save one image every 1 second
        if current_time - last_save_time >= save_interval:

            filename = os.path.join(
                save_folder,
                f"{class_name.lower()}_{collected}.jpg"
            )

            saved_ok = cv2.imwrite(filename, scanning_area)

            if saved_ok:
                collected += 1
                last_save_time = current_time
                failed_saves = 0
                print(f"Saved image {collected}")

            else:
                # Give up instead of looping forever if the file cannot be saved
                failed_saves += 1
                last_save_time = current_time
                print("The image could not be saved.")

                if failed_saves >= 3:
                    print("Saving keeps failing. Check the folder, then try again.")
                    break

        # Draw the green scanning box
        cv2.rectangle(
            frame,
            (x1, y1),
            (x2, y2),
            (0, 255, 0),
            3
        )

        # Show the number of collected images
        cv2.putText(
            frame,
            f"{class_name}: {collected}/{target}",
            (x1, y1 - 15),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 0),
            2
        )

        # Show the live video in Jupyter Notebook
        jpeg_ok, jpeg = cv2.imencode(".jpg", frame)

        if jpeg_ok:

            if handle is None:
                handle = display(
                    Image(data=jpeg.tobytes()),
                    display_id=True
                )

            else:
                handle.update(
                    Image(data=jpeg.tobytes())
                )

        time.sleep(0.03)

    camera.release()

    print(f"Finished. The folder now has {collected} {class_name} images.")
    print("Camera released.")

In [ ]:
collect_images("LEFT", LEFT_FOLDER, total_images=30)

In [ ]:
collect_images("RIGHT", RIGHT_FOLDER, total_images=30)

In [ ]:
collect_images("STOP", STOP_FOLDER, total_images=30)

In [ ]:
collect_images("NO_SIGN", NO_SIGN_FOLDER, total_images=30)

3. check the data set

In [ ]:
import glob

left_images = glob.glob(LEFT_FOLDER + "/*.jpg")
right_images = glob.glob(RIGHT_FOLDER + "/*.jpg")
stop_images = glob.glob(STOP_FOLDER + "/*.jpg")
no_sign_images = glob.glob(NO_SIGN_FOLDER + "/*.jpg")

print(f"LEFT: {len(left_images)}")
print(f"RIGHT: {len(right_images)}")
print(f"STOP: {len(stop_images)}")
print(f"NO_SIGN: {len(no_sign_images)}")

In [ ]:
minimum_images = 28

dataset_ready = True

if len(left_images) < minimum_images:
    print("LEFT needs more images.")
    dataset_ready = False

if len(right_images) < minimum_images:
    print("RIGHT needs more images.")
    dataset_ready = False

if len(stop_images) < minimum_images:
    print("STOP needs more images.")
    dataset_ready = False

if len(no_sign_images) < minimum_images:
    print("NO_SIGN needs more images.")
    dataset_ready = False

if dataset_ready:
    print("The dataset is ready for training.")
else:
    print("Please collect more images before training.")

4. Turn one picture into KNN numbers

In [ ]:
import cv2
import numpy as np
import glob
import os
import time
from IPython.display import display, Image

# The size KNN will use for every picture
IMAGE_SIZE = 32

# How many neighbours are allowed to vote
K = 3

BOX_SIZE = 300


def prepare_image(image):
    # Step 1: Change the image to grayscale
    gray = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2GRAY
    )

    # Step 2: Make every image 32 x 32
    small_image = cv2.resize(
        gray,
        (IMAGE_SIZE, IMAGE_SIZE)
    )

    # Step 3: Change the square into one long row
    pixel_row = small_image.flatten()

    # Step 4: Change the numbers to float32
    pixel_row = pixel_row.astype(
        np.float32
    )

    # Step 5: Change 0-255 into 0-1
    pixel_row = pixel_row / 255.0

    return pixel_row


print("prepare_image is ready.")
print("Every picture will become", IMAGE_SIZE * IMAGE_SIZE, "numbers.")

5. Load the training images

In [ ]:
if not dataset_ready:
    print("The dataset is not ready. Collect more images first.")

else:
    training_data = []
    training_labels = []

    # LEFT = 0
    for filename in glob.glob(os.path.join(LEFT_FOLDER, "*.jpg")):
        image = cv2.imread(filename)

        if image is not None:
            training_data.append(prepare_image(image))
            training_labels.append(0)

    # RIGHT = 1
    for filename in glob.glob(os.path.join(RIGHT_FOLDER, "*.jpg")):
        image = cv2.imread(filename)

        if image is not None:
            training_data.append(prepare_image(image))
            training_labels.append(1)

    # STOP = 2
    for filename in glob.glob(os.path.join(STOP_FOLDER, "*.jpg")):
        image = cv2.imread(filename)

        if image is not None:
            training_data.append(prepare_image(image))
            training_labels.append(2)

    # NO_SIGN = 3
    for filename in glob.glob(os.path.join(NO_SIGN_FOLDER, "*.jpg")):
        image = cv2.imread(filename)

        if image is not None:
            training_data.append(prepare_image(image))
            training_labels.append(3)

    # Change the Python lists into NumPy arrays for OpenCV
    training_data = np.array(
        training_data,
        dtype=np.float32
    )

    training_labels = np.array(
        training_labels,
        dtype=np.float32
    ).reshape(-1, 1)

    print("Training data shape  :", training_data.shape)
    print("Training labels shape:", training_labels.shape)

6. Train the KNN model

In [ ]:
knn = cv2.ml.KNearest_create()

knn.train(
    training_data,
    cv2.ml.ROW_SAMPLE,
    training_labels
)

print("KNN training is complete.")

7. Predict one picture

In [ ]:
def predict_sign(image):
    pixel_row = prepare_image(image)

    # OpenCV wants one sample row
    pixel_row = pixel_row.reshape(1, -1)

    _, result, neighbours, distances = knn.findNearest(
        pixel_row,
        K
    )

    label = int(result[0][0])

    if label == 0:
        prediction = "LEFT"

    elif label == 1:
        prediction = "RIGHT"

    elif label == 2:
        prediction = "STOP"

    else:
        prediction = "NO_SIGN"

    return prediction


print("predict_sign is ready.")

8. KNN test without the UGOT

In [ ]:
# TEST VERSION - no robot, no try/except.
# Run sections 4 to 7 first.

import cv2
import time
from IPython.display import display, Image

# The test stops by itself after this many seconds
MAX_SECONDS = 60

# Open the MacBook camera
camera = cv2.VideoCapture(0, cv2.CAP_AVFOUNDATION)

# Windows:
# camera = cv2.VideoCapture(0, cv2.CAP_DSHOW)

# Do NOT set the camera resolution here. See the warning in section 4.

prediction_history = []
handle = None

last_action = None
message = "Hold the sign steady"

if not camera.isOpened():
    print("The camera could not be opened.")

else:
    print("Show a sign to the camera.")
    print("Show STOP to finish, or wait", MAX_SECONDS, "seconds.")

    start_time = time.time()

    while time.time() - start_time < MAX_SECONDS:

        frame_ok, frame = camera.read()

        if not frame_ok:
            print("The camera could not capture an image.")
            break

        # Build the same 300 x 300 box used during collection
        frame_height, frame_width = frame.shape[:2]

        x1 = (frame_width - BOX_SIZE) // 2
        y1 = (frame_height - BOX_SIZE) // 2
        x2 = x1 + BOX_SIZE
        y2 = y1 + BOX_SIZE

        # Crop before drawing the green box
        scanning_area = frame[y1:y2, x1:x2]

        # Ask KNN about this one frame
        prediction = predict_sign(scanning_area)

        # Keep only the latest 5 predictions
        prediction_history.append(prediction)

        if len(prediction_history) > 5:
            prediction_history.pop(0)

        # Count the votes
        left_votes = prediction_history.count("LEFT")
        right_votes = prediction_history.count("RIGHT")
        stop_votes = prediction_history.count("STOP")
        no_sign_votes = prediction_history.count("NO_SIGN")

        # A result needs at least 3 votes out of 5
        stable_result = "WAITING"

        if left_votes >= 3:
            stable_result = "LEFT"

        elif right_votes >= 3:
            stable_result = "RIGHT"

        elif stop_votes >= 3:
            stable_result = "STOP"

        elif no_sign_votes >= 3:
            stable_result = "NO_SIGN"

        # The STOP sign ends the test
        if stable_result == "STOP":
            print("STOP sign recognised. The test is finished.")
            break

        # Print the message only when the action changes
        if stable_result != last_action:

            if stable_result == "LEFT":
                message = "Left turn"

            elif stable_result == "RIGHT":
                message = "Right turn"

            elif stable_result == "NO_SIGN":
                message = "Moving forward"

            else:
                message = "Hold the sign steady"

            print(message)
            last_action = stable_result

        # Draw the green box, the messages and the vote counts
        cv2.rectangle(
            frame,
            (x1, y1),
            (x2, y2),
            (0, 255, 0),
            3
        )

        cv2.putText(
            frame,
            f"KNN: {prediction}",
            (20, 35),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.75,
            (0, 255, 255),
            2
        )

        cv2.putText(
            frame,
            message,
            (20, 75),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.75,
            (0, 255, 0),
            2
        )

        cv2.putText(
            frame,
            f"L:{left_votes} R:{right_votes} S:{stop_votes} N:{no_sign_votes}",
            (20, 115),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 255, 255),
            2
        )

        # Show the live video in Jupyter Notebook
        jpeg_ok, jpeg = cv2.imencode(".jpg", frame)

        if jpeg_ok:

            if handle is None:
                handle = display(
                    Image(data=jpeg.tobytes()),
                    display_id=True
                )

            else:
                handle.update(
                    Image(data=jpeg.tobytes())
                )

        time.sleep(0.03)

camera.release()
print("Camera released.")

9. Decide what the UGOT should do

In [ ]:
# Rotation speed, degrees per second, allowed range 5 to 280
TURN_SPEED = 20

# Forward speed, centimetres per second, allowed range 5 to 80
FORWARD_SPEED = 15


def do_action(stable_result):
    if stable_result == "LEFT":
        # 2 means turn left
        got.mecanum_turn_speed(2, TURN_SPEED)
        return "ACTION: TURN LEFT"

    elif stable_result == "RIGHT":
        # 3 means turn right
        got.mecanum_turn_speed(3, TURN_SPEED)
        return "ACTION: TURN RIGHT"

    elif stable_result == "NO_SIGN":
        # 0 means forward
        got.mecanum_move_speed(0, FORWARD_SPEED)
        return "ACTION: FORWARD"

    else:
        # Not sure yet, so do not move
        got.mecanum_stop()
        return "HOLD THE SIGN STEADY"


print("do_action is ready.")

10. Live recognition and robot control

In [ ]:
# Open the MacBook camera
camera = cv2.VideoCapture(0, cv2.CAP_AVFOUNDATION)

# Windows:
# camera = cv2.VideoCapture(0, cv2.CAP_DSHOW)

# Do NOT set the camera resolution here. See the warning in section 4.

prediction_history = []
handle = None

# Remember the last command sent to the robot
last_action = None
message = "HOLD THE SIGN STEADY"

if not camera.isOpened():
    print("The camera could not be opened.")
    camera.release()

else:
    print("Place the robot on the floor with space in front of it.")

    for number in range(3, 0, -1):
        print(number)
        time.sleep(1)

    print("Started. Show the STOP sign to finish.")

    try:
        while True:

            frame_ok, frame = camera.read()

            if not frame_ok:
                print("The camera could not capture an image.")
                break

            # Build the same 300 x 300 box used during collection
            frame_height, frame_width = frame.shape[:2]

            x1 = (frame_width - BOX_SIZE) // 2
            y1 = (frame_height - BOX_SIZE) // 2
            x2 = x1 + BOX_SIZE
            y2 = y1 + BOX_SIZE

            # Crop before drawing the green box, so the box is not in the picture
            scanning_area = frame[y1:y2, x1:x2]

            # Ask KNN about this one frame
            prediction = predict_sign(scanning_area)

            # Keep only the latest 5 predictions
            prediction_history.append(prediction)

            if len(prediction_history) > 5:
                prediction_history.pop(0)

            # Count the votes
            left_votes = prediction_history.count("LEFT")
            right_votes = prediction_history.count("RIGHT")
            stop_votes = prediction_history.count("STOP")
            no_sign_votes = prediction_history.count("NO_SIGN")

            # A result needs at least 3 votes out of 5
            stable_result = "WAITING"

            if left_votes >= 3:
                stable_result = "LEFT"

            elif right_votes >= 3:
                stable_result = "RIGHT"

            elif stop_votes >= 3:
                stable_result = "STOP"

            elif no_sign_votes >= 3:
                stable_result = "NO_SIGN"

            # The STOP sign ends the whole program
            if stable_result == "STOP":
                got.mecanum_stop()
                print("STOP sign recognised. The program is finished.")
                got.play_audio_tts("Stop sign detected. Mission complete.", 0, False)
                break

            # Only send a command when the action CHANGES.
            # The UGOT keeps doing the last thing it was told, so sending the
            # same command on every frame would flood the robot with messages.
            if stable_result != last_action:
                message = do_action(stable_result)
                last_action = stable_result

            # Draw the green box and the two messages
            cv2.rectangle(
                frame,
                (x1, y1),
                (x2, y2),
                (0, 255, 0),
                3
            )

            cv2.putText(
                frame,
                f"KNN: {prediction}",
                (20, 35),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.75,
                (0, 255, 255),
                2
            )

            cv2.putText(
                frame,
                message,
                (20, 75),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.75,
                (0, 255, 0),
                2
            )

            # Show the live video in Jupyter Notebook
            jpeg_ok, jpeg = cv2.imencode(".jpg", frame)

            if jpeg_ok:

                if handle is None:
                    handle = display(
                        Image(data=jpeg.tobytes()),
                        display_id=True
                    )

                else:
                    handle.update(
                        Image(data=jpeg.tobytes())
                    )

            time.sleep(0.03)

    except KeyboardInterrupt:
        print("Stopped by hand.")

    finally:
        got.mecanum_stop()
        camera.release()
        print("The wheels are stopped and the camera is released.")